# Credit Card Fraud Detection with CyborgDB

This demo shows how to use CyborgDB's encrypted cosine similarity search for fraud detection.

**Method:** Threshold-based K-Nearest Neighbors - flag a transaction as fraud if ANY of its k nearest neighbors is fraudulent.

## Setup and Data Loading

In [ ]:
%pip install --quiet --upgrade cyborgdb-core>=0.13.0 kagglehub pandas numpy scikit-learn

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# Download dataset
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
csv_path = os.path.join(path, "creditcard.csv")
df = pd.read_csv(csv_path)

print(f"Dataset: {df.shape[0]:,} transactions")
print(f"Fraud rate: {df['Class'].mean()*100:.3f}%")

# Prepare data
X = df.drop('Class', axis=1).values
y = df['Class'].values

# Normalize features
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining: {len(X_train):,} samples")
print(f"Testing: {len(X_test):,} samples")

## Setup CyborgDB with In-Memory Storage

In [ ]:
import getpass
import os
from cyborgdb_core import Client, DBConfig, get_demo_api_key

# Configure API key or demo API key (expires in 1 hour)
API_KEY = os.environ.get("CYBORGDB_API_KEY") or get_demo_api_key()
os.environ["CYBORGDB_API_KEY"] = API_KEY

# Create client with in-memory storage
client = Client(
    API_KEY,
    DBConfig(location="memory"),
    DBConfig(location="memory"),
    DBConfig(location="memory")
)

print("✓ CyborgDB configured with in-memory storage")

## Create Index and Upsert Training Data

In [ ]:
import secrets
import uuid

# Generate a secure 32-byte key for encryption
DEMO_INDEX_KEY = secrets.token_bytes(32)

# Create encrypted index
index_name = f"fraud_demo_{uuid.uuid4().hex[:8]}"
index = client.create_index(index_name, DEMO_INDEX_KEY)
print(f"Created index: {index_name}")

# Upsert training data
print(f"\nUpserting {len(X_train):,} training samples...")
batch_docs = [
    {
        "id": f"train_{i}",
        "vector": X_train[i].tolist(),
        "metadata": {"is_fraud": int(y_train[i])}
    }
    for i in range(len(X_train))
]

index.upsert(batch_docs)
print(f"✓ Upserted {len(X_train):,} training samples")

## Fraud Detection Function

In [ ]:
def detect_fraud(test_vectors, k=10, fraud_threshold=1, batch_size=50):
    """
    Detect fraud using CyborgDB similarity search.
    
    Args:
        test_vectors: Transaction feature vectors
        k: Number of nearest neighbors to check
        fraud_threshold: Minimum fraud neighbors to flag as suspicious
        batch_size: Batch size for queries
    
    Returns:
        predictions: Binary predictions (0=normal, 1=fraud)
        fraud_counts: Number of fraud neighbors found for each transaction
    """
    predictions = []
    fraud_counts = []
    
    print(f"Analyzing {len(test_vectors):,} transactions (k={k}, threshold={fraud_threshold})...")
    
    for i in tqdm(range(0, len(test_vectors), batch_size)):
        batch_end = min(i + batch_size, len(test_vectors))
        batch_vectors = test_vectors[i:batch_end]
        
        # Query CyborgDB for nearest neighbors
        batch_results = index.query(batch_vectors.tolist(), top_k=k)
        
        for results in batch_results:
            if results:
                # Count fraud neighbors
                fraud_count = sum(
                    1 for r in results 
                    if r.get('metadata', {}).get('is_fraud', 0) == 1
                )
                fraud_counts.append(fraud_count)
                predictions.append(1 if fraud_count >= fraud_threshold else 0)
            else:
                fraud_counts.append(0)
                predictions.append(0)
    
    return np.array(predictions), np.array(fraud_counts)

## Run Fraud Detection and Evaluate

In [ ]:
# Create balanced test set (all fraud cases + sample of normal)
fraud_indices = np.where(y_test == 1)[0]
normal_indices = np.where(y_test == 0)[0]

n_fraud = len(fraud_indices)
n_normal = min(n_fraud * 10, len(normal_indices))

np.random.seed(42)
sampled_normal = np.random.choice(normal_indices, n_normal, replace=False)
test_indices = np.concatenate([fraud_indices, sampled_normal])
np.random.shuffle(test_indices)

X_test_eval = X_test[test_indices]
y_test_eval = y_test[test_indices]

print(f"Test set: {len(X_test_eval)} transactions ({np.sum(y_test_eval == 1)} fraud, {np.sum(y_test_eval == 0)} normal)\n")

# Run fraud detection
y_pred, fraud_neighbor_counts = detect_fraud(X_test_eval, k=10, fraud_threshold=1, batch_size=50)

## Results

In [ ]:
# Classification report
print("\n" + "="*70)
print("FRAUD DETECTION RESULTS")
print("="*70)
print("\nClassification Report:")
print(classification_report(y_test_eval, y_pred, target_names=['Normal', 'Fraud'], zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test_eval, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print(f"                  Predicted Normal  Predicted Fraud")
print(f"Actual Normal:    {tn:>15}  {fp:>15}")
print(f"Actual Fraud:     {fn:>15}  {tp:>15}")

# Calculate metrics
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*70)
print("KEY METRICS")
print("="*70)
print(f"Fraud Detection Rate (Recall):  {recall*100:>6.2f}%  ← Caught {tp}/{tp+fn} fraud cases")
print(f"Precision:                       {precision*100:>6.2f}%  ← {fp} false positives")
print(f"F1 Score:                        {f1*100:>6.2f}%")
print(f"Overall Accuracy:                {((tp+tn)/(tp+tn+fp+fn))*100:>6.2f}%")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"✓ Detected {tp} out of {tp+fn} fraudulent transactions ({recall*100:.1f}%)")
print(f"✓ Only {fp} false alarm(s) out of {tn+fp} normal transactions")
print(f"✓ {precision*100:.1f}% of fraud alerts were correct")

## Example: Analyze a Fraud Case

In [ ]:
# Find a correctly detected fraud case
fraud_cases = np.where((y_test_eval == 1) & (y_pred == 1))[0]

if len(fraud_cases) > 0:
    idx = fraud_cases[0]
    
    print("\n" + "="*70)
    print(f"EXAMPLE: Analyzing transaction #{idx}")
    print("="*70)
    print(f"Actual label: FRAUD")
    print(f"Predicted: FRAUD ✓")
    print(f"Fraud neighbors found: {fraud_neighbor_counts[idx]}/10")
    
    # Get nearest neighbors
    results = index.query(X_test_eval[idx].tolist(), top_k=10)
    
    if results:
        print(f"\nTop 10 nearest neighbors:")
        for i, r in enumerate(results, 1):
            is_fraud = r[0].get('metadata', {}).get('is_fraud', 0)
            distance = r[0].get('distance', 0)
            label = 'FRAUD' if is_fraud else 'NORMAL'
            print(f"  {i:2}. {r[0].get('id'):>15} - {label:6} (distance: {distance:>7.4f})")
        
        print(f"\n→ Similar to {fraud_neighbor_counts[idx]} known fraud cases → Flagged as FRAUD")